# 06 – MuRIL Fine-Tuning

## Objective
Fine-tune MuRIL (a BERT-family model pretrained on Indian languages)
for 3-class sentiment classification on the cleaned Daraz Nepal reviews.

## Input
- Cleaned splits from notebook 02: `train_cleaned.csv`, `val_cleaned.csv`, `test_cleaned.csv`

## Output
- Fine-tuned model saved to `muril_final_model_v2` on Google Drive
- Final verified test macro F1: 0.6779

In [ ]:
!pip install transformers datasets torch

In [ ]:
import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/research_transformer/train_cleaned.csv')
val_df = pd.read_csv('/content/drive/MyDrive/research_transformer/val_cleaned.csv')
test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(3041, 7)
(650, 7)
(652, 7)


In [ ]:
print(train_df[['review_text', 'sentiment_label']].head())
print(train_df['sentiment_label'].unique())

                                         review_text sentiment_label
0                              Bluetooth not working        negative
1  To be very honest I didn’t like the smell it w...        positive
2  My Honest Review of Boom 141 Earbuds\r\nI’m sh...        positive
3            Ramro xa paisa anusar thikai ho product        positive
4  The item matched the image precisely, and the ...        positive
['negative' 'positive' 'neutral']


In [ ]:
# Same mapping applied to all three splits, so numbers mean the same thing everywhere
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}

train_df['label'] = train_df['sentiment_label'].map(label_map)
val_df['label'] = val_df['sentiment_label'].map(label_map)
test_df['label'] = test_df['sentiment_label'].map(label_map)

# Quick check
print(train_df[['sentiment_label', 'label']].head())
print(train_df['label'].value_counts())

  sentiment_label  label
0        negative      0
1        positive      2
2        positive      2
3        positive      2
4        positive      2
label
2    1390
1     830
0     821
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# Quick test on one real review from your data
sample = train_df['review_text'].iloc[3]  # "Ramro xa paisa anusar thikai ho product"
tokens = tokenizer(sample, truncation=True, padding='max_length', max_length=128)

print(sample)
print(tokens['input_ids'][:20])  # just first 20 numbers to keep it readable

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Ramro xa paisa anusar thikai ho product
[104, 15372, 6395, 195, 1192, 29538, 138415, 14806, 2735, 1926, 8143, 105, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
train_encodings = tokenizer(list(train_df['review_text']), truncation=True, padding='max_length', max_length=128)
val_encodings = tokenizer(list(val_df['review_text']), truncation=True, padding='max_length', max_length=128)
test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding='max_length', max_length=128)

print("Train encoded:", len(train_encodings['input_ids']))
print("Val encoded:", len(val_encodings['input_ids']))
print("Test encoded:", len(test_encodings['input_ids']))

Train encoded: 3041
Val encoded: 650
Test encoded: 652


In [ ]:
import torch

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, list(train_df['label']))
val_dataset = ReviewDataset(val_encodings, list(val_df['label']))
test_dataset = ReviewDataset(test_encodings, list(test_df['label']))

print(train_dataset[0])

{'input_ids': tensor([  104, 75302,  1235,  4504,   105,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

## First attempt: standard (unweighted) fine-tuning
This first run uses plain cross-entropy loss, no class weighting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import f1_score

# 1. Load data
train_df = pd.read_csv('/content/drive/MyDrive/research_transformer/train_cleaned.csv')
val_df = pd.read_csv('/content/drive/MyDrive/research_transformer/val_cleaned.csv')
test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')

# 2. Map labels to numbers
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label'] = train_df['sentiment_label'].map(label_map)
val_df['label'] = val_df['sentiment_label'].map(label_map)
test_df['label'] = test_df['sentiment_label'].map(label_map)

# 3. Tokenize
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
train_encodings = tokenizer(list(train_df['review_text']), truncation=True, padding='max_length', max_length=128)
val_encodings = tokenizer(list(val_df['review_text']), truncation=True, padding='max_length', max_length=128)
test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding='max_length', max_length=128)

# 4. Build Dataset objects
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, list(train_df['label']))
val_dataset = ReviewDataset(val_encodings, list(val_df['label']))
test_dataset = ReviewDataset(test_encodings, list(test_df['label']))

# 5. Load model
model = AutoModelForSequenceClassification.from_pretrained("google/muril-base-cased", num_labels=3)

# 6. Training setup
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/research_transformer/muril_results',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {"f1_macro": macro_f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Setup complete. Ready to train.")

Mounted at /content/drive


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Setup complete. Ready to train.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,No log,0.965021,0.449144
2,No log,0.893127,0.452766
3,0.928022,0.870946,0.470035
4,0.928022,0.862960,0.472148


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=764, training_loss=0.8747321922741635, metrics={'train_runtime': 424.8333, 'train_samples_per_second': 28.632, 'train_steps_per_second': 1.798, 'total_flos': 800127903310848.0, 'train_loss': 0.8747321922741635, 'epoch': 4.0})

The unweighted run above caused the neutral class to collapse
(very low recall, since it's the hardest and most ambiguous class).
This retrain adds class weights `[1.5, 1.7, 1.0]` (negative, neutral,
positive) to push the model to pay more attention to the harder classes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1. Load data
train_df = pd.read_csv('/content/drive/MyDrive/research_transformer/train_cleaned.csv')
val_df = pd.read_csv('/content/drive/MyDrive/research_transformer/val_cleaned.csv')
test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')

# 2. Map labels to numbers
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label'] = train_df['sentiment_label'].map(label_map)
val_df['label'] = val_df['sentiment_label'].map(label_map)
test_df['label'] = test_df['sentiment_label'].map(label_map)

# 3. Tokenize
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
train_encodings = tokenizer(list(train_df['review_text']), truncation=True, padding='max_length', max_length=128)
val_encodings = tokenizer(list(val_df['review_text']), truncation=True, padding='max_length', max_length=128)
test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding='max_length', max_length=128)

# 4. Build Dataset objects
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, list(train_df['label']))
val_dataset = ReviewDataset(val_encodings, list(val_df['label']))
test_dataset = ReviewDataset(test_encodings, list(test_df['label']))

# 5. Load fresh model
model = AutoModelForSequenceClassification.from_pretrained("google/muril-base-cased", num_labels=3)

# 6. Class weights (negative, neutral, positive) — neutral gets highest weight since it's hardest
class_weights = torch.tensor([1.5, 1.7, 1.0]).to(model.device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# 7. Metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {"f1_macro": macro_f1}

# 8. Training setup — saving to LOCAL disk, not Drive, to avoid quota issues
training_args = TrainingArguments(
    output_dir='/content/muril_results_v2',
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 9. Train
trainer.train()

# 10. Detailed results
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print(classification_report(true_labels, preds, target_names=['negative', 'neutral', 'positive']))
print(confusion_matrix(true_labels, preds))

Mounted at /content/drive


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    negative       0.64      0.65      0.65       176
     neutral       0.49      0.57      0.53       177
    positive       0.84      0.74      0.79       297

    accuracy                           0.67       650
   macro avg       0.66      0.66      0.65       650
weighted avg       0.69      0.67      0.68       650

[[115  50  11]
 [ 45 101  31]
 [ 20  56 221]]


In [ ]:
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

print(classification_report(test_true, test_preds, target_names=['negative', 'neutral', 'positive']))
print(confusion_matrix(test_true, test_preds))

              precision    recall  f1-score   support

    negative       0.66      0.68      0.67       176
     neutral       0.53      0.57      0.55       178
    positive       0.85      0.80      0.82       298

    accuracy                           0.70       652
   macro avg       0.68      0.68      0.68       652
weighted avg       0.71      0.70      0.71       652

[[120  47   9]
 [ 45 101  32]
 [ 17  44 237]]


In [ ]:
trainer.save_model('/content/drive/MyDrive/research_transformer/muril_final_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Verifying the saved model
Reloading the model saved above, from a fresh session, to confirm
it was saved correctly before relying on it further.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report

# 1. Load data (same files, same mapping as original training)
train_df = pd.read_csv('/content/drive/MyDrive/research_transformer/train_cleaned.csv')
val_df = pd.read_csv('/content/drive/MyDrive/research_transformer/val_cleaned.csv')
test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label'] = train_df['sentiment_label'].map(label_map)
val_df['label'] = val_df['sentiment_label'].map(label_map)
test_df['label'] = test_df['sentiment_label'].map(label_map)

# 2. Load the FINE-TUNED model directly (not the base checkpoint)
MODEL_PATH = '/content/drive/MyDrive/research_transformer/muril_final_model'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# 3. Tokenize exactly as in original training (padding=True, not 'max_length')
def tokenize_texts(texts, max_length=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

test_encodings = tokenize_texts(test_df['review_text'])

# 4. Same ReviewDataset class as original notebook
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

test_dataset = ReviewDataset(test_encodings, test_df['label'].tolist())

# 5. Bare-bones trainer just for prediction (no training args needed)
trainer = Trainer(model=model)

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

report = classification_report(
    test_true, test_preds,
    target_names=['negative', 'neutral', 'positive'],
    digits=4
)
print(report)
print("Macro F1:", f1_score(test_true, test_preds, average='macro'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    negative     0.0000    0.0000    0.0000       176
     neutral     0.2686    0.9551    0.4192       178
    positive     0.3158    0.0201    0.0379       298

    accuracy                         0.2699       652
   macro avg     0.1948    0.3251    0.1524       652
weighted avg     0.2177    0.2699    0.1318       652

Macro F1: 0.1523634671012796


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import os
from datetime import datetime

MODEL_PATH = '/content/drive/MyDrive/research_transformer/muril_final_model'

print("Files in folder:")
for f in os.listdir(MODEL_PATH):
    full_path = os.path.join(MODEL_PATH, f)
    size_mb = os.path.getsize(full_path) / (1024 * 1024)
    mod_time = datetime.fromtimestamp(os.path.getmtime(full_path))
    print(f"  {f}: {size_mb:.1f} MB, last modified {mod_time}")

print("\nModel config:")
print(model.config.id2label)
print("Num labels:", model.config.num_labels)

print("\nClassifier weight statistics (sanity check):")
classifier_weights = model.classifier.weight.data if hasattr(model, 'classifier') else model.classifier.out_proj.weight.data
print("Mean:", classifier_weights.mean().item())
print("Std:", classifier_weights.std().item())
print("Min/Max:", classifier_weights.min().item(), classifier_weights.max().item())

Files in folder:
  config.json: 0.0 MB, last modified 2026-08-27 15:02:06
  model.safetensors: 906.2 MB, last modified 2026-08-27 15:02:17
  training_args.bin: 0.0 MB, last modified 2026-08-27 15:02:18

Model config:
{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2'}
Num labels: 3

Classifier weight statistics (sanity check):
Mean: -0.0004634953511413187
Std: 0.022527627646923065
Min/Max: -0.07311869412660599 0.07667281478643417


The reload above showed the classifier had an untrained head
(weight std ~0.02, the default random-initialization value) despite
training completing normally , the save had captured the wrong model
state. This cell retrains from scratch and includes a built-in check
(classifier weight std) immediately after training, before saving,
so the same issue can't slip through again.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# 1. Load data
train_df = pd.read_csv('/content/drive/MyDrive/research_transformer/train_cleaned.csv')
val_df = pd.read_csv('/content/drive/MyDrive/research_transformer/val_cleaned.csv')
test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')

# 2. Map labels to numbers
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_df['label'] = train_df['sentiment_label'].map(label_map)
val_df['label'] = val_df['sentiment_label'].map(label_map)
test_df['label'] = test_df['sentiment_label'].map(label_map)

# 3. Tokenize
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
train_encodings = tokenizer(list(train_df['review_text']), truncation=True, padding='max_length', max_length=128)
val_encodings = tokenizer(list(val_df['review_text']), truncation=True, padding='max_length', max_length=128)
test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding='max_length', max_length=128)

# 4. Build Dataset objects
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = ReviewDataset(train_encodings, list(train_df['label']))
val_dataset = ReviewDataset(val_encodings, list(val_df['label']))
test_dataset = ReviewDataset(test_encodings, list(test_df['label']))

# 5. Load fresh model
model = AutoModelForSequenceClassification.from_pretrained("google/muril-base-cased", num_labels=3)

# 6. Class weights
class_weights = torch.tensor([1.5, 1.7, 1.0]).to(model.device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {"f1_macro": macro_f1}

training_args = TrainingArguments(
    output_dir='/content/muril_results_v2',
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 9. Train
trainer.train()

# ============================================================
# EVERYTHING BELOW RUNS IMMEDIATELY, SAME CELL, NO GAP
# ============================================================

# 10. Sanity check the classifier head is actually trained (should NOT be ~0.02)
classifier_std = trainer.model.classifier.weight.data.std().item()
print(f"\nClassifier weight std after training: {classifier_std:.4f} (should be well above 0.02)\n")

# 11. Evaluate on TEST set (not val) — this is the number that goes in the paper
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

print("=== TEST SET RESULTS ===")
print(classification_report(test_true, test_preds, target_names=['negative', 'neutral', 'positive'], digits=4))
print("Macro F1:", f1_score(test_true, test_preds, average='macro'))
print(confusion_matrix(test_true, test_preds))

# 12. Save immediately, right here, no gap
SAVE_PATH = '/content/drive/MyDrive/research_transformer/muril_final_model_v2'
trainer.save_model(SAVE_PATH)
print(f"\nSaved to {SAVE_PATH}")

# 13. Immediately reload and re-verify in this same session
reload_check = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
reload_std = reload_check.classifier.weight.data.std().item()
print(f"Reloaded classifier std: {reload_std:.4f} (should match step 10's value, NOT ~0.02)")

Mounted at /content/drive


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

=== TEST SET RESULTS ===
              precision    recall  f1-score   support

    negative     0.6941    0.6705    0.6821       176
     neutral     0.5412    0.5169    0.5287       178
    positive     0.8045    0.8423    0.8230       298

    accuracy                         0.7071       652
   macro avg     0.6799    0.6765    0.6779       652
weighted avg     0.7028    0.7071    0.7046       652

Macro F1: 0.6779224589038435
[[118  46  12]
 [ 37  92  49]
 [ 15  32 251]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved to /content/drive/MyDrive/research_transformer/muril_final_model_v2


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reloaded classifier std: 0.0246 (should match step 10's value, NOT ~0.02)


## Tokenizer loading check
A second issue: loading the tokenizer from the saved model folder
(instead of the original Hugging Face Hub source) silently breaks
tokenization and drops performance. The next cell shows this happening,
followed by the corrected version.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import f1_score
import csv

# Load the VERIFIED model (the _v2 one, properly trained)
MODEL_PATH = '/content/drive/MyDrive/research_transformer/muril_final_model_v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
test_df['label'] = test_df['sentiment_label'].map(label_map)

test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding=True, max_length=128, return_tensors="pt")

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

test_dataset = ReviewDataset(test_encodings, list(test_df['label']))
trainer = Trainer(model=model)

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

macro_f1_value = f1_score(test_true, test_preds, average='macro')
print("MuRIL Macro F1:", macro_f1_value)

with open('/content/drive/MyDrive/research_transformer/transformer_results.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["MuRIL", macro_f1_value])

print("Saved to transformer_results.csv")

Mounted at /content/drive


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

MuRIL Macro F1: 0.26291802638551864
Saved to transformer_results.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import f1_score
import csv

MODEL_PATH = '/content/drive/MyDrive/research_transformer/muril_final_model_v2'

# Load tokenizer from the ORIGINAL source, not from the saved folder
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")
# Load the fine-tuned MODEL WEIGHTS from your saved folder
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

test_df = pd.read_csv('/content/drive/MyDrive/research_transformer/test_cleaned.csv')
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
test_df['label'] = test_df['sentiment_label'].map(label_map)

test_encodings = tokenizer(list(test_df['review_text']), truncation=True, padding=True, max_length=128, return_tensors="pt")

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

test_dataset = ReviewDataset(test_encodings, list(test_df['label']))
trainer = Trainer(model=model)

test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_true = test_predictions.label_ids

macro_f1_value = f1_score(test_true, test_preds, average='macro')
print("MuRIL Macro F1:", macro_f1_value)

with open('/content/drive/MyDrive/research_transformer/transformer_results.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["MuRIL", macro_f1_value])

print("Saved to transformer_results.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

MuRIL Macro F1: 0.6779224589038435
Saved to transformer_results.csv
